# Trivima — Multi-View Room Generation AI

Train a Diffusion Transformer to generate 8 consistent room views from 1 photo.

**Pipeline:** Single photo → Multi-View DiT → 8 views → 3DGS → Photorealistic 3D

**Data:** RealEstate10K (real room walk-through videos with camera poses)

**Training Strategy:**
- Phase 1: 10% data → validate model learns
- Phase 2: 40% data → measure improvement
- Phase 3: 100% data → production model

## Step 1: Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Install dependencies
!pip install -q torch torchvision gdown tqdm Pillow numpy

In [ ]:
# Clone Trivima repo (or upload files)
!git clone https://github.com/Harishmelvic/trivima.git /content/trivima 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, '/content/trivima')
sys.path.insert(0, '/content')

## Step 2: Download RealEstate10K Dataset

In [ ]:
# Download the pixelSplat preprocessed RE10K subset
import gdown
import os

os.makedirs('/content/data', exist_ok=True)

# Small subset (600MB) - enough for Phase 1
gdown.download_folder(
    'https://drive.google.com/drive/folders/1joiezNCyQK2BvWMnfwHJpm2V77c7iYGe',
    output='/content/data/re10k_raw'
)

print('Downloaded!')
!ls -lh /content/data/re10k_raw/

In [ ]:
# Extract
import zipfile

with zipfile.ZipFile('/content/data/re10k_raw/re10k_subset.zip') as zf:
    zf.extractall('/content/data/re10k_extracted')

print('Extracted!')

In [ ]:
# Convert to training format
import torch
import numpy as np
import json
import io
import glob
from PIL import Image

DATA_ROOT = '/content/data/re10k_extracted/re10k_subset'
OUTPUT_DIR = '/content/data/realestate10k/train'
NUM_VIEWS = 8
IMG_SIZE = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)
pair_count = 0
total_scenes = 0

for split in ['train', 'test']:
    torch_files = sorted(glob.glob(os.path.join(DATA_ROOT, split, '*.torch')))
    for tf in torch_files:
        scenes = torch.load(tf, map_location='cpu', weights_only=False)
        for scene in scenes:
            n_frames = len(scene['images'])
            if n_frames < NUM_VIEWS + 1:
                continue
            total_scenes += 1

            images = []
            for img_bytes in scene['images']:
                img = Image.open(io.BytesIO(img_bytes.numpy().tobytes())).convert('RGB')
                images.append(img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS))

            cameras = scene['cameras'].numpy()

            step = max(1, n_frames // 3)
            for pair_idx in range(min(3, n_frames // (NUM_VIEWS + 1))):
                input_idx = pair_idx * step
                remaining = [i for i in range(n_frames) if i != input_idx]
                view_step = max(1, len(remaining) // NUM_VIEWS)
                target_idx = remaining[::view_step][:NUM_VIEWS]

                if len(target_idx) < NUM_VIEWS:
                    continue

                pair_dir = os.path.join(OUTPUT_DIR, f'scene_{total_scenes:04d}', f'pair_{pair_idx:03d}')
                os.makedirs(pair_dir, exist_ok=True)

                images[input_idx].save(os.path.join(pair_dir, 'input.png'))

                poses = []
                for vi, ti in enumerate(target_idx):
                    images[ti].save(os.path.join(pair_dir, f'target_{vi:02d}.png'))
                    ext = cameras[ti][6:18].reshape(3, 4)
                    pose = np.eye(4)
                    pose[:3, :4] = ext
                    poses.append(pose.tolist())

                with open(os.path.join(pair_dir, 'cameras.json'), 'w') as f:
                    json.dump({'poses': poses}, f)

                pair_count += 1

print(f'Total: {total_scenes} scenes, {pair_count} training pairs')

## Step 3: Model Architecture

In [ ]:
# Copy model files (if not cloned from git)
# Upload trivima/multiview/model.py, dataset.py, train.py

# Or define inline:
from trivima.multiview.model import MultiViewDiT
from trivima.multiview.train import DDPMScheduler
from trivima.multiview.dataset import RealEstate10KDataset

# Quick test
model = MultiViewDiT(num_views=8, img_size=128, embed_dim=512, depth=8).cuda()
print(f'Model: {model.count_params()/1e6:.1f}M params')

# Test forward pass
photo = torch.randn(1, 3, 128, 128).cuda()
noisy = torch.randn(1, 8, 3, 128, 128).cuda()
t = torch.tensor([500.0]).cuda()
poses = torch.eye(4).unsqueeze(0).unsqueeze(0).expand(1, 8, -1, -1).cuda()
out = model(photo, noisy, t, poses)
print(f'Output shape: {out.shape} — PASS!')

## Step 4: Train Phase 1 (10% data)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import json

device = 'cuda'
NUM_VIEWS = 8
IMG_SIZE = 128
BATCH_SIZE = 2  # Increase to 4 on A100
EPOCHS = 50
LR = 1e-4

# Model
model = MultiViewDiT(
    num_views=NUM_VIEWS, img_size=IMG_SIZE,
    embed_dim=512, depth=8
).to(device)

# Data
dataset = RealEstate10KDataset('/content/data/realestate10k', num_target_views=NUM_VIEWS, img_size=IMG_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

print(f'Training: {len(dataset)} pairs, {len(loader)} batches/epoch')

# Training
scheduler = DDPMScheduler().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

os.makedirs('/content/checkpoints', exist_ok=True)
best_loss = float('inf')
history = []

for epoch in range(EPOCHS):
    t0 = time.time()
    losses = []
    model.train()

    for batch_idx, batch in enumerate(loader):
        inp = batch['input'].to(device)
        targets = batch['targets'].to(device)
        poses = batch['poses'].to(device)
        B = inp.shape[0]

        t = torch.randint(0, 1000, (B,), device=device)
        noise = torch.randn_like(targets)
        noisy = scheduler.add_noise(targets, noise, t)

        pred = model(inp, noisy, t.float(), poses)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())

        if (batch_idx + 1) % 20 == 0:
            print(f'  [{epoch+1}/{EPOCHS}] batch {batch_idx+1}/{len(loader)} loss={np.mean(losses[-20:]):.4f}')

    avg = np.mean(losses)
    dt = time.time() - t0
    history.append({'epoch': epoch+1, 'loss': avg})
    print(f'Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f} ({dt:.1f}s)')

    if avg < best_loss:
        best_loss = avg
        torch.save({'model': model.state_dict(), 'loss': best_loss}, '/content/checkpoints/best.pt')
        print(f'  -> New best: {best_loss:.4f}')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')

## Step 5: Evaluate — Generate Multi-View

In [ ]:
# Load best model
ckpt = torch.load('/content/checkpoints/best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded model, best loss: {ckpt["loss"]:.4f}')

# Pick a test scene
test_batch = next(iter(loader))
input_img = test_batch['input'][:1].to(device)
gt_targets = test_batch['targets'][:1].to(device)
poses = test_batch['poses'][:1].to(device)

# Generate views via denoising
x = torch.randn(1, NUM_VIEWS, 3, IMG_SIZE, IMG_SIZE, device=device)
n_steps = 50
step_size = 1000 // n_steps

with torch.no_grad():
    for i in range(999, -1, -step_size):
        t = torch.full((1,), i, device=device, dtype=torch.float)
        noise_pred = model(input_img, x, t, poses)
        alpha_t = scheduler.alpha_cumprod[i]
        alpha_prev = scheduler.alpha_cumprod[max(0, i - step_size)]
        pred_x0 = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
        pred_x0 = pred_x0.clamp(-1, 1)
        if i > 0:
            x = torch.sqrt(alpha_prev) * pred_x0 + torch.sqrt(1 - alpha_prev) * torch.randn_like(x)
        else:
            x = pred_x0

print('Generated', NUM_VIEWS, 'views!')

In [ ]:
# Visualize: Input | Generated views | Ground truth
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, NUM_VIEWS + 1, figsize=(20, 8))

# Row 0: Input (repeated)
inp_np = input_img[0].cpu().numpy().transpose(1, 2, 0)
for j in range(NUM_VIEWS + 1):
    if j == 0:
        axes[0, j].imshow(inp_np)
        axes[0, j].set_title('Input')
    else:
        axes[0, j].axis('off')
axes[0, 0].set_ylabel('Input')

# Row 1: Generated
for j in range(NUM_VIEWS):
    gen = ((x[0, j].cpu().numpy().transpose(1, 2, 0) + 1) / 2).clip(0, 1)
    axes[1, j + 1].imshow(gen)
    axes[1, j + 1].set_title(f'Gen {j}')
axes[1, 0].set_ylabel('Generated')
axes[1, 0].axis('off')

# Row 2: Ground truth
for j in range(NUM_VIEWS):
    gt = ((gt_targets[0, j].cpu().numpy().transpose(1, 2, 0) + 1) / 2).clip(0, 1)
    axes[2, j + 1].imshow(gt)
    axes[2, j + 1].set_title(f'GT {j}')
axes[2, 0].set_ylabel('Ground Truth')
axes[2, 0].axis('off')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Multi-View Generation: Input → Generated → Ground Truth')
plt.tight_layout()
plt.savefig('/content/multiview_results.png', dpi=150)
plt.show()
print('Results saved to /content/multiview_results.png')

In [ ]:
# Compute PSNR
import torch.nn.functional as F

gen_views = (x + 1) / 2  # [-1,1] -> [0,1]
gt_views = (gt_targets + 1) / 2

mse = F.mse_loss(gen_views, gt_views).item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
print(f'MSE: {mse:.4f}')
print(f'PSNR: {psnr:.2f} dB')
print(f'\nPhase 1 target: PSNR > 18 dB')
print(f'Result: {"PASS" if psnr > 18 else "NEEDS MORE DATA/TRAINING"}')

## Step 6: Save Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
os.makedirs('/content/drive/MyDrive/trivima_checkpoints', exist_ok=True)
shutil.copy('/content/checkpoints/best.pt', '/content/drive/MyDrive/trivima_checkpoints/multiview_phase1.pt')
print('Checkpoint saved to Google Drive!')

## Next Steps

If PSNR > 18 dB → proceed to Phase 2 with 40% data:
1. Download full RE10K from `http://schadenfreude.csail.mit.edu:8000/`
2. Train for 200K iterations
3. Evaluate cross-view consistency

If PSNR < 15 dB → need architecture changes:
- Increase model size (768 embed, 12 depth)
- Add perceptual loss
- Use pre-trained image encoder (DINOv2)